# H001 - Linear probing of the residual stream

**Question.** Is a simple concept (sentiment) a *linearly decodable direction* in a
language model's residual stream, and if so, at what depth does it become readable?

This notebook runs end to end in **synthetic mode** (no GPU, no model download) so
you can see the whole pipeline and the figure immediately. Flip one flag to run a
real **Gemma 3 1B** capture instead.

The teaching point is the *shape* of the accuracy-by-layer curve, plus the two
controls that separate a real interpretability claim from a fooled one:
a **shuffled-label control task** (selectivity) and a **bag-of-tokens baseline**.

## 1. Setup

From the repo root: `pip install -e ".[dev]"`. The cell below makes the `src/`
package importable whether or not you installed it.

In [ ]:
import sys, pathlib
# make src/ importable when running from the sample directory without installing
root = pathlib.Path.cwd()
for up in [root, *root.parents]:
    if (up / "src" / "mechinterp_samples").exists():
        sys.path.insert(0, str(up / "src")); break

from mechinterp_samples import (
    SentimentDataset, ActivationCapturer, MeanPooler,
    LayerProbeSweep, LayerCurvePlotter,
)
import numpy as np
print("imports ok")

## 2. The dataset

`SentimentDataset` builds short polar-adjective prompts. The split is
**vocabulary-disjoint**: a quarter of the adjectives in each class are held out, so
test prompts use words never seen in training. That blocks a probe from memorising
exact words, though (as we will see in the real run) it does *not* block sentiment
leaking through embedding space.

In [ ]:
dataset = SentimentDataset(seed=0)
split = dataset.build()
print(f"n_train={split.n_train}  n_test={split.n_test}")
print("train example:", split.texts_for("train")[0])
print("test  example:", split.texts_for("test")[0])

## 3. Get activations

Set `USE_REAL = True` to capture genuine Gemma 3 1B residual-stream activations
(needs the `capture` extra, a GPU, and accepted access to the gated model). Left
`False`, we synthesise activations with a concept that **strengthens with depth**,
which is what a genuinely *computed* concept would look like.

Note the discipline encoded in `ActivationCapturer`: load via HuggingFace
`transformers` with `output_hidden_states` (never Ollama), in bf16 (never quantize
activations you intend to interpret). `hidden_states[0]` is the embedding output;
layers `1..n` are post-block residual streams.

In [ ]:
USE_REAL = False  # flip to True for a real Gemma 3 1B capture

def synthetic_activations(split, n_layers=12, d_model=64, seed=0):
    rng = np.random.default_rng(seed)
    y = split.labels; n = len(y)
    direction = rng.normal(size=d_model)
    acts = {}
    for layer in range(n_layers):
        strength = 2.5 * (layer / (n_layers - 1))  # 0 at embedding -> strong deep
        acts[layer] = rng.normal(size=(n, d_model)) + (y[:, None] * strength) * direction
    return acts

if USE_REAL:
    capturer = ActivationCapturer("google/gemma-3-1b-it", pooler=MeanPooler())
    acts = capturer.capture(split.texts)
    model_label = "google/gemma-3-1b-it"
else:
    acts = synthetic_activations(split, seed=0)
    model_label = "synthetic"

print(f"captured {len(acts)} layers, d_model={acts[0].shape[1]}, model={model_label}")

## 4. Probe every layer, with controls

`LayerProbeSweep` trains a `LinearProbe` (logistic regression) at each layer and,
crucially, a **control** probe of identical capacity on *shuffled* labels.
**Selectivity** = real held-out accuracy minus control accuracy. It also computes a
**bag-of-tokens** baseline: if the residual-stream probe cannot beat logistic
regression on raw token counts, we have only shown the *words* differ, not that the
*representation* encodes the concept.

In [ ]:
report = LayerProbeSweep(seed=0).run(acts, split, concept=dataset.name, model=model_label)

print(f"chance            : {report.chance_acc:.3f}")
print(f"bag-of-tokens     : {report.baseline_test_acc:.3f}")
print(f"best layer        : {report.best_layer.layer} (test {report.best_layer.test_acc:.3f})")
bs = report.best_selectivity_layer
print(f"best selectivity  : layer {bs.layer}  sel {bs.selectivity:.3f} "
      f"(test {bs.test_acc:.3f}, control {bs.control_test_acc:.3f})")

## 5. The figure

The *shape* is the argument. A concept the model **computes** should be hard to
read at layer 0 and rise with depth. A concept that is merely **lexically present**
is readable from the embedding.

In [ ]:
fig_path = LayerCurvePlotter().plot(report, "figures/h001_accuracy_by_layer.png")
report.to_json("figures/h001_report.json")

from IPython.display import Image
Image(filename=str(fig_path))

## 6. Reading the result

**Synthetic mode (above).** Held-out accuracy starts near chance at layer 0 and
climbs to ~1.0 by the middle layers, while the shuffled-label control stays near
chance. That rise-with-depth profile is what a genuinely *computed* concept looks
like.

**Real Gemma 3 1B (`USE_REAL = True`).** The honest result is different and is the
reason this sample exists: accuracy is near-perfect at **every** layer, *including
layer 0, the raw embedding before any transformer computation*. Polar adjectives
are linearly separable in embedding space, so the probe succeeds without the model
computing anything. A vocabulary-disjoint split stops word-memorisation but not
embedding-space leakage.

**Takeaway.** The apparatus works and sentiment *is* linearly decodable, but this
setup does not localise *where the model computes* sentiment. The fix (a follow-up
demo) is a concept that **cannot** be read off the embedding, e.g. one requiring
composition across several tokens, so depth has to do real work. Fuller research
notes are available on request.